<a href="https://colab.research.google.com/github/jjkiljanski/biebrza-shrub-encroachment-analysis/blob/main/notebooks/biebrza_streaming_1997_2015_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Streaming Conv1D Encroachment Prediction from Google Earth Engine (1997–2015)

This notebook:

1. Initializes Google Earth Engine and Google Drive in Colab.
2. Builds a 10-step (1997–2015) biannual Landsat time-series image (6 bands per step).
3. Creates a **tile index** over the area of interest (AOI).
4. Streams each tile from GEE → runs your Conv1D model → saves prediction tiles to Google Drive.
5. Is **resumable**: if Colab disconnects, simply rerun the notebook and it will continue from the next unprocessed tile.

The model expects input of shape `(batch, T, C)` with:
- `T = 10` time steps (1997–2015 biannual)
- `C = 6` bands (`NDMI, NBR, NIR, NDVI, SWIR1, SWIR2`)
- `num_classes = 5`

Predictions are saved as **one-band GeoTIFF tiles** with the predicted class index (0–4) in Google Drive.
You can later mosaic these prediction tiles into a full-park encroachment risk map.

In [1]:
# Install required packages (Colab)
!pip install -q earthengine-api geemap rasterio torch torchvision tqdm shapely

In [2]:
import os
import json
import math

import ee
import geemap
import torch
import torch.nn as nn
import rasterio
from rasterio.transform import from_origin
import numpy as np
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive (for prediction tiles + tile index)
drive.mount('/content/drive')

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='biebrza-encroachment-analysis')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load stats defining column naming and normalization during model traing

In [3]:
import pandas as pd

# Path to the normalization stats you saved from the training notebook
norm_stats_path = "/content/drive/MyDrive/GEE_Biebrza/norm_stats_biannual_6bands.csv"

norm_df = pd.read_csv(norm_stats_path)
print("Loaded norm stats with", len(norm_df), "features")
print(norm_df.head())

# The columns in norm_df['col'] define the feature order the model expects.
original_ts_cols = norm_df["col"].tolist()
mean_vec = norm_df["mean"].values.astype("float32")  # shape (B,)
std_vec  = norm_df["std"].values.astype("float32")   # shape (B,)

Loaded norm stats with 60 features
             col      mean       std
0  NBR_1997_1998  0.266412  0.051863
1  NBR_1999_2000  0.276463  0.055741
2  NBR_2001_2002  0.310012  0.051301
3  NBR_2003_2004  0.306458  0.045374
4  NBR_2005_2006  0.299410  0.053837


## Define Area of Interest (AOI) and Build 1997–2015 Time-Series Image

In [4]:
import ipywidgets as widgets
from datetime import datetime

# ============================================================
# Load Biebrzański National Park boundary
#    (from the WDPA – World Database on Protected Areas)
# ============================================================

wdpa = ee.FeatureCollection("WCMC/WDPA/current/polygons")

# Filter areas whose NAME contains "Biebrza" (safe way to match spelling)
bpn = wdpa.filter(ee.Filter.stringContains("NAME", "Biebrza"))

print("Number of matching park polygons:", bpn.size().getInfo())

# Geometry for clipping satellite images
aoi = bpn.geometry()

print('AOI bounds (lon/lat):', aoi.bounds().coordinates().getInfo())

Number of matching park polygons: 3
AOI bounds (lon/lat): [[[22.39636562061044, 53.19907326752765], [23.597246250349325, 53.19907326752765], [23.597246250349325, 53.80556002887963], [22.39636562061044, 53.80556002887963], [22.39636562061044, 53.19907326752765]]]


In [5]:
# Build 1997–2015 biannual image stack (10 steps x 6 bands = 60 bands)
project_root = 'projects/biebrza-encroachment-analysis/assets/image_composites'
bands = ['NDMI', 'NBR', 'NIR', 'NDVI', 'SWIR1', 'SWIR2']

# Biannual years for Window A: 1997, 1999, ..., 2015
windowA_years = list(range(1997, 2016, 2))  # 10 time steps

def load_biannual_image(year):
    asset_id = f"{project_root}/biannual_{year}_{year+1}"
    img = ee.Image(asset_id).select(bands)
    # IMPORTANT: match training columns, e.g. "NDMI_1997_1998"
    rename_list = [f"{b}_{year}_{year+1}" for b in bands]
    renamed = img.rename(rename_list)
    return renamed

# Build the stacked image
images = [load_biannual_image(y) for y in windowA_years]
stack_img = ee.Image.cat(images).clip(aoi)

# Reorder bands to match training column order exactly
windowA_img = stack_img.select(original_ts_cols)

# Inspect projection & scale
proj = windowA_img.projection()
crs = proj.crs().getInfo()
scale = proj.nominalScale().getInfo()

print('Window A years:', windowA_years)
print('CRS:', crs)
print('Scale (m):', scale)
print('Number of bands:', windowA_img.bandNames().size().getInfo())
print('First 10 EE bands:', windowA_img.bandNames().slice(0, 10).getInfo())
print('First 10 original_ts_cols :', original_ts_cols[:10])

Window A years: [1997, 1999, 2001, 2003, 2005, 2007, 2009, 2011, 2013, 2015]
CRS: EPSG:4326
Scale (m): 10
Number of bands: 60
First 10 EE bands: ['NBR_1997_1998', 'NBR_1999_2000', 'NBR_2001_2002', 'NBR_2003_2004', 'NBR_2005_2006', 'NBR_2007_2008', 'NBR_2009_2010', 'NBR_2011_2012', 'NBR_2013_2014', 'NBR_2015_2016']
First 10 ts_cols : ['NBR_1997_1998', 'NBR_1999_2000', 'NBR_2001_2002', 'NBR_2003_2004', 'NBR_2005_2006', 'NBR_2007_2008', 'NBR_2009_2010', 'NBR_2011_2012', 'NBR_2013_2014', 'NBR_2015_2016']


## Create / Load Tile Index (Resumable)

We tile the AOI into ~256×256-pixel patches at the native scale, and store a JSON file in Drive
tracking which tiles are **done**. If Colab disconnects, you just rerun the notebook and it will
continue from the next unprocessed tile.

Tile JSON schema:
```json
{
  "meta": {"tile_pixels": 256, "scale": 30, ...},
  "tiles": [
    {"id": 0, "lon_min": ..., "lat_min": ..., "lon_max": ..., "lat_max": ..., "done": false},
    ...
  ]
}
```

In [6]:
from shapely.geometry import shape, box

pred_base_dir = '/content/drive/MyDrive/biebrza_preds'
os.makedirs(pred_base_dir, exist_ok=True)

tile_json_path = os.path.join(pred_base_dir, 'windowA_tiles.json')
tile_pred_dir = os.path.join(pred_base_dir, 'windowA_tiles')
os.makedirs(tile_pred_dir, exist_ok=True)

TILE_PIXELS = 256  # ~256x256 pixel tiles

def create_tile_index(image, aoi, tile_pixels, json_path):
    """Create a tile index over the AOI and save it to JSON in Drive.
    Tiles are defined in lon/lat (EPSG:4326) as rectangles, but we KEEP ONLY
    those that actually intersect the AOI geometry, using shapely locally.
    """
    # --- Pull AOI once as GeoJSON and convert to shapely ---
    aoi_geo = aoi.getInfo()   # should be a dict with 'type' and 'coordinates'
    # If you want to inspect it once, you can uncomment:
    # print(aoi_geo.keys(), aoi_geo.get('type', None))
    aoi_geom = shape(aoi_geo)  # THIS is the key change

    # AOI bounds in lon/lat (still using EE here)
    bounds = aoi.bounds().coordinates().getInfo()[0]
    lons = [pt[0] for pt in bounds]
    lats = [pt[1] for pt in bounds]
    min_lon, max_lon = min(lons), max(lons)
    min_lat, max_lat = min(lats), max(lats)

    # Center latitude for lon-degree calculation
    center_lat = 0.5 * (min_lat + max_lat)
    center_lat_rad = math.radians(center_lat)

    # Use image scale to approximate tile size in meters
    proj = image.projection()
    pixel_scale = proj.nominalScale().getInfo()  # meters per pixel
    tile_size_m = tile_pixels * pixel_scale

    # Convert meters to degrees (approximate)
    meters_per_deg_lat = 111320.0
    meters_per_deg_lon = meters_per_deg_lat * math.cos(center_lat_rad)

    lat_step = tile_size_m / meters_per_deg_lat
    lon_step = tile_size_m / meters_per_deg_lon if meters_per_deg_lon != 0 else tile_size_m / meters_per_deg_lat

    tiles = []
    tile_id = 0
    num_candidates = 0
    num_kept = 0

    lat = min_lat
    while lat < max_lat:
        next_lat = min(lat + lat_step, max_lat)
        lon = min_lon
        while lon < max_lon:
            next_lon = min(lon + lon_step, max_lon)
            num_candidates += 1

            # shapely rectangle for this tile
            tile_poly = box(lon, lat, next_lon, next_lat)

            # local intersection test (no EE call here)
            if tile_poly.intersects(aoi_geom):
                tiles.append({
                    'id': tile_id,
                    'lon_min': lon,
                    'lat_min': lat,
                    'lon_max': next_lon,
                    'lat_max': next_lat,
                    'done': False,
                })
                num_kept += 1

            tile_id += 1
            lon = next_lon
        lat = next_lat

    state = {
        'meta': {
            'tile_pixels': tile_pixels,
            'pixel_scale_m': pixel_scale,
            'crs': proj.crs().getInfo(),
            'aoi_bounds': {
                'min_lon': min_lon,
                'max_lon': max_lon,
                'min_lat': min_lat,
                'max_lat': max_lat,
            },
            'num_candidates': num_candidates,
            'num_kept': num_kept,
        },
        'tiles': tiles,
    }

    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(state, f, indent=2)

    print(
        f'Created tile index with {len(tiles)} tiles '
        f'(kept {num_kept} / {num_candidates} candidates) '
        f'and saved to {json_path}'
    )
    return state

# ---- load or create tile index ----
if os.path.exists(tile_json_path):
    print('Loading existing tile index from:', tile_json_path)
    with open(tile_json_path, 'r', encoding='utf-8') as f:
        tiles_state = json.load(f)
else:
    print('No existing tile index found. Creating a new one...')
    tiles_state = create_tile_index(windowA_img, aoi, TILE_PIXELS, tile_json_path)

tiles = tiles_state['tiles']
num_tiles = len(tiles)
num_done = sum(1 for t in tiles if t.get('done'))
print(f'Total tiles intersecting AOI: {num_tiles}, already done: {num_done}')

No existing tile index found. Creating a new one...
Created tile index with 291 tiles (kept 291 / 864 candidates) and saved to /content/drive/MyDrive/biebrza_preds/windowA_tiles.json
Total tiles intersecting AOI: 291, already done: 0


## Load Conv1D Model

This cell loads the Conv1D classifier you trained. Make sure `conv1d_best_model.pth`
is uploaded to `/content/` in this Colab session (or change `model_path`).

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

class Conv1DClassifier(nn.Module):
    def __init__(self, seq_len, num_classes, in_channels):
        super().__init__()
        self.seq_len = seq_len
        self.conv1 = nn.Conv1d(in_channels=in_channels, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32,        out_channels=64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)  # global average pooling over time
        self.fc   = nn.Linear(64, num_classes)

    def forward(self, x):
        # x: [B, T, C]
        x = x.permute(0, 2, 1)   # [B, C, T]
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)  # [B, 64]
        logits = self.fc(x)
        return logits

T = 10
C = 6
num_classes = 5

# Path to your trained weights (upload this file to /content first)
model_path = '/content/conv1d_best_model.pth'

model = Conv1DClassifier(seq_len=T, num_classes=num_classes, in_channels=C)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

print('Model loaded successfully!')

Using device: cpu
Model loaded successfully!


### Load yearly normalization stats for each index

## Helper Functions: Download Tile from GEE, Run Model, Save Prediction

Each tile is processed as follows:
1. Use `geemap.ee_export_image` to download a small multi-band GeoTIFF tile from GEE to `/content`.
2. Read it with `rasterio` into a `(B, H, W)` NumPy array (with `B = 60` bands).
3. Reshape to `(H·W, T, C)` and run the Conv1D model in batches.
4. Take `argmax` over classes to get the predicted class index per pixel (0–4).
5. Save the prediction tile as a one-band GeoTIFF to Google Drive.

The tile's `done` flag is then set to `True` and the JSON index is updated
so that the process can be **resumed** later.

In [8]:
def download_tile_from_ee(tile, image, scale, tmp_dir='/content'):
    """
    Download a small tile from GEE as a GeoTIFF and return (data, profile),
    or (None, None) if the tile is effectively empty.

    Returns:
      data: np.ndarray (bands, H, W) [float32]
      profile: rasterio profile
    """
    lon_min, lat_min = tile['lon_min'], tile['lat_min']
    lon_max, lat_max = tile['lon_max'], tile['lat_max']
    tile_id = tile['id']

    geom = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])
    tmp_path = os.path.join(tmp_dir, f'windowA_tile_{tile_id:05d}.tif')

    print(f"\n--- Downloading tile {tile_id} ---")

    # Export a small tile (should be well under EE's request-size limit)
    geemap.ee_export_image(
        image,
        filename=tmp_path,
        scale=scale,
        region=geom,
        file_per_band=False,
    )

    if not os.path.exists(tmp_path):
        print(f"  ❌ Failed to download tile {tile_id} to {tmp_path}")
        return None, None

    with rasterio.open(tmp_path) as src:
        data = src.read().astype("float32")  # (bands, H, W)
        profile = src.profile
        nodata = src.nodata

    # Clean up input tile to save space
    try:
        os.remove(tmp_path)
    except OSError:
        pass

    # ---- RAW DATA SANITY CHECKS ----
    B, H, W = data.shape
    total_px = data.size

    # NaN stats
    nan_count = np.isnan(data).sum()
    nan_pct = 100.0 * nan_count / total_px
    print(f"  Tile {tile_id}: raw data shape = {data.shape}, NaN% = {nan_pct:.2f}%")

    # All-nodata check
    if nodata is not None:
        if np.all(data == nodata):
            print(f"  ⚠️ Tile {tile_id} is ALL nodata ({nodata}). Skipping.")
            return None, None

    # Constant tile check
    if np.all(data == data.flat[0]):
        print(f"  ⚠️ Tile {tile_id} is constant value ({data.flat[0]:.4f}). Skipping.")
        return None, None

    # Dynamic range
    dmin = float(np.nanmin(data))
    dmax = float(np.nanmax(data))
    print(f"  Tile {tile_id}: raw min={dmin:.4f}, max={dmax:.4f}")

    return data, profile


def run_model_on_tile_array(
    tile_data,
    model,
    T=10,
    C=6,
    num_classes=5,
    inner_batch_size=4096,
    mean_vec=None,
    std_vec=None,
):
    """
    Run the Conv1D model on a tile array with per-feature normalization.

    tile_data: numpy array (B, H, W) with B = T * C = 60
    mean_vec, std_vec: arrays of shape (B,) with train-time mean/std for each feature.

    Returns three numpy arrays of shape (H, W):
      - dominant_class: argmax over classes (float32, values 0..4)
      - p_encroachment: probability of class 'wetland_to_woody' (index 4)
      - uncertainty: 1 - max_class_probability
    """
    B, H, W = tile_data.shape
    expected_B = T * C
    if B != expected_B:
        raise ValueError(f'Expected {expected_B} bands, got {B}')

    # ---- 1. Normalize per feature (same as training) ----
    if mean_vec is not None and std_vec is not None:
        if mean_vec.shape[0] != B or std_vec.shape[0] != B:
            raise ValueError(
                f'mean/std length mismatch: got {mean_vec.shape[0]} stats for {B} bands'
            )
        flat = tile_data.reshape(B, -1)  # (B, H*W)
        flat_norm = (flat - mean_vec[:, None]) / std_vec[:, None]
        tile_data = flat_norm.reshape(B, H, W)

    # ---- 2. Reshape for Conv1D (same as before) ----
    tile_tc_hw = tile_data.reshape(T, C, H, W)          # (T, C, H, W)
    tile_hw_tc = np.transpose(tile_tc_hw, (2, 3, 0, 1)) # (H, W, T, C)
    N = H * W
    tile_n_tc = tile_hw_tc.reshape(N, T, C)             # (N, T, C)

    x = torch.from_numpy(tile_n_tc).float().to(device)

    all_pred_classes = []
    all_max_probs = []
    all_enc_probs = []

    encroachment_class_idx = 4  # wetland_to_woody

    model.eval()
    with torch.no_grad():
        for start in range(0, N, inner_batch_size):
            end = min(start + inner_batch_size, N)
            batch = x[start:end]                 # (batch_size, T, C)
            logits = model(batch)                # (batch_size, num_classes)
            probs = torch.softmax(logits, dim=1) # (batch_size, num_classes)

            max_probs, pred_classes = probs.max(dim=1)           # (batch_size,)
            enc_probs = probs[:, encroachment_class_idx]         # (batch_size,)

            all_pred_classes.append(pred_classes.cpu())
            all_max_probs.append(max_probs.cpu())
            all_enc_probs.append(enc_probs.cpu())

    # Concatenate all batches
    all_pred_classes = torch.cat(all_pred_classes, dim=0).numpy()  # (N,)
    all_max_probs    = torch.cat(all_max_probs, dim=0).numpy()     # (N,)
    all_enc_probs    = torch.cat(all_enc_probs, dim=0).numpy()     # (N,)

    # Reshape back to (H, W)
    dominant_class = all_pred_classes.reshape(H, W).astype("float32")
    p_encroachment = all_enc_probs.reshape(H, W).astype("float32")
    max_prob_hw    = all_max_probs.reshape(H, W).astype("float32")
    uncertainty    = 1.0 - max_prob_hw

    return dominant_class, p_encroachment, uncertainty


def process_single_tile(tile, image, model, scale, pred_dir,
                        T=10, C=6, num_classes=5, inner_batch_size=4096,
                        mean_vec=None, std_vec=None):
    """
    Download a tile, run the model, and save a 3-band prediction GeoTIFF:

      band 1: dominant class (0..4, float32)
      band 2: p(wetland_to_woody)
      band 3: uncertainty = 1 - max_class_prob
    """
    tile_id = tile['id']
    print(f"\n=== Processing tile {tile_id} ===")

    tile_data, profile = download_tile_from_ee(tile, image, scale)
    if tile_data is None:
        print(f"  Tile {tile_id} has no valid data. Marking as done without prediction.")
        return None

    dom_class, p_enc, uncertainty = run_model_on_tile_array(
        tile_data,
        model,
        T=T,
        C=C,
        num_classes=num_classes,
        inner_batch_size=inner_batch_size,
        mean_vec=mean_vec,
        std_vec=std_vec,
    )

    # ---- MODEL OUTPUT SANITY CHECKS ----
    print(f"  Tile {tile_id}: class_min={np.nanmin(dom_class)}, class_max={np.nanmax(dom_class)}")
    print(f"  Tile {tile_id}: p_enc min/max = {np.nanmin(p_enc):.4f}/{np.nanmax(p_enc):.4f}")
    print(f"  Tile {tile_id}: uncertainty min/max = {np.nanmin(uncertainty):.4f}/{np.nanmax(uncertainty):.4f}")

    if np.all(np.isnan(p_enc)):
        print(f"  ⚠️ WARNING: Tile {tile_id} → ALL NaN encroachment probabilities!")

    if np.nanmax(p_enc) < 0.01:
        print(f"  ⚠️ WARNING: Tile {tile_id} → extremely low encroachment probabilities")

    if np.nanmax(dom_class) == np.nanmin(dom_class):
        print(f"  ⚠️ WARNING: Tile {tile_id} → constant class predictions ({dom_class[0, 0]})")

    # Safety: clean NaNs if any
    dom_class   = np.nan_to_num(dom_class,  nan=0.0)
    p_enc       = np.nan_to_num(p_enc,      nan=0.0)
    uncertainty = np.nan_to_num(uncertainty, nan=1.0)

    out_profile = profile.copy()
    out_profile.update(
        driver='GTiff',
        count=3,
        dtype='float32',
        compress='lzw',
    )

    pred_path = os.path.join(pred_dir, f'pred_windowA_tile_{tile_id:05d}.tif')
    with rasterio.open(pred_path, 'w', **out_profile) as dst:
        dst.write(dom_class.astype('float32'), 1)
        dst.write(p_enc.astype('float32'), 2)
        dst.write(uncertainty.astype('float32'), 3)

    print(f"  ✅ Saved prediction tile to {pred_path}")
    return pred_path


## Main Processing Loop (Resumable)

This will:
1. Iterate over all tiles in the JSON index.
2. Skip tiles where `done == True`.
3. For each remaining tile: download → predict → save prediction to Drive.
4. Mark `tile['done'] = True` and write the updated JSON back to Drive after each tile.

If Colab disconnects, just rerun all cells above **and then rerun this cell**.
The loop will continue from the next unfinished tile.

In [9]:
num_tiles = len(tiles)
num_done = sum(1 for t in tiles if t.get('done'))
print(f'Starting processing loop. Total tiles: {num_tiles}, already done: {num_done}')

for tile in tiles:
    if tile.get('done'):
        continue

    try:
        _ = process_single_tile(
            tile=tile,
            image=windowA_img,
            model=model,
            scale=scale,
            pred_dir=tile_pred_dir,
            T=T,
            C=C,
            num_classes=num_classes,
            inner_batch_size=4096,
            mean_vec=mean_vec,
            std_vec=std_vec,
        )
        tile['done'] = True
    except Exception as e:
        print(f'Error processing tile {tile["id"]}: {e}')
    finally:
        with open(tile_json_path, 'w', encoding='utf-8') as f:
            json.dump(tiles_state, f, indent=2)


num_done = sum(1 for t in tiles if t.get('done'))
print(f'Processing finished (or loop ended). Tiles done: {num_done} / {num_tiles}')

Starting processing loop. Total tiles: 291, already done: 0

=== Processing tile 0 ===

--- Downloading tile 0 ---
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00000.tif
  Tile 0: raw data shape = (60, 257, 431), NaN% = 0.00%
  Tile 0: raw min=-0.0878, max=26177.0000
  Tile 0: class_min=0.0, class_max=4.0
  Tile 0: p_enc min/max = 0.0000/0.9507
  Tile 0: uncertainty min/max = 0.0008/0.7410
  ✅ Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00000.tif

=== Processing tile 1 ===

--- Downloading tile 1 ---
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00001.tif
  Tile 1: raw data shape = (60, 257, 432), NaN% = 0.00%
  Tile 1: raw min=-0.1055, max=26740.5000
  Tile 1: class_min=0.0, class_max=4.0
  Tile 1: p_enc min/max = 0.0000/0.8912
  Tile 1: uncertainty min/max = 0.0002/0.7230
  ✅ Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00001.tif

=== Processing tile 2 ===

--- Downloading tile 2 ---
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00002.tif
  Tile 2: raw data shape = (60, 257, 431), NaN% = 0.00%
  Tile 2: raw min=-0.0873, max=26243.0000
  Tile 2: class_min=0.0, class_max=4.0
  Tile 2: p_enc min/max = 0.0000/0.9140
  Tile 2: uncertainty min/max = 0.0013/0.7621
  ✅ Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00002.tif

=== Processing tile 3 ===

--- Downloading tile 3 ---
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00003.tif
  Tile 3: raw data shape = (60, 257, 431), NaN% = 0.00%
  Tile 3: raw min=-0.1144, max=25952.5000
  Tile 3: class_min=0.0, class_max=4.0
  Tile 3: p_enc min/max = 0.0000/0.9047
  Tile 3: uncertainty min/max = 0.0000/0.7494
  ✅ Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00003.tif

=== Processing tile 4 ===

--- Downloading tile 4 ---
Generating URL ...
Please wait ...


...

Data downloaded to /content/windowA_tile_00787.tif
  Tile 787: raw data shape = (60, 257, 432), NaN% = 0.00%
  Tile 787: raw min=-0.1002, max=25411.5000
  Tile 787: class_min=0.0, class_max=4.0
  Tile 787: p_enc min/max = 0.0000/0.9443
  Tile 787: uncertainty min/max = 0.0009/0.7578
  ✅ Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00787.tif

=== Processing tile 788 ===

--- Downloading tile 788 ---
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00788.tif
  Tile 788: raw data shape = (60, 257, 431), NaN% = 0.00%
  Tile 788: raw min=-0.1069, max=25068.5000
  Tile 788: class_min=0.0, class_max=4.0
  Tile 788: p_enc min/max = 0.0000/0.9781
  Tile 788: uncertainty min/max = 0.0008/0.7404
  ✅ Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00788.tif

=== Processing tile 813 ===

--- Downloading tile 813 ---
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00813.tif
  Tile 813: raw data shape = (60, 257, 431), NaN% = 0.00%
  Tile 813: raw min=-0.0621, max=26170.0000
  Tile 813: class_min=0.0, class_max=4.0
  Tile 813: p_enc min/max = 0.0000/0.9553
  Tile 813: uncertainty min/max = 0.0066/0.7311
  ✅ Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00813.tif

=== Processing tile 814 ===

--- Downloading tile 814 ---
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00814.tif
  Tile 814: raw data shape = (60, 257, 432), NaN% = 0.00%
  Tile 814: raw min=-0.0601, max=27468.5000
  Tile 814: class_min=0.0, class_max=4.0
  Tile 814: p_enc min/max = 0.0000/0.9332
  Tile 814: uncertainty min/max = 0.0028/0.7638
  ✅ Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00814.tif

=== Processing tile 846 ===

--- Downloading tile 846 ---
Generating URL ...
Please wait ...


Data downloaded to /content/windowA_tile_00846.tif
  Tile 846: raw data shape = (60, 96, 432), NaN% = 0.00%
  Tile 846: raw min=-0.0398, max=25448.0000
  Tile 846: class_min=0.0, class_max=4.0
  Tile 846: p_enc min/max = 0.0003/0.8968
  Tile 846: uncertainty min/max = 0.0111/0.7286
  ✅ Saved prediction tile to /content/drive/MyDrive/biebrza_preds/windowA_tiles/pred_windowA_tile_00846.tif
Processing finished (or loop ended). Tiles done: 291 / 291


## (Final Post-Prediction Step) Mosaic Prediction Tiles into a Single Raster

Once all tiles are processed (`done == True` for all), you can merge them into a single
encroachment map. This step can be memory-intensive depending on your AOI size, so
you may want to run it on a smaller AOI first (e.g., Lower Biebrza).

In [ ]:
from glob import glob
from rasterio.merge import merge

# Read all prediction tiles
pred_tile_paths = sorted(glob(os.path.join(tile_pred_dir, 'pred_windowA_tile_*.tif')))
print(f'Found {len(pred_tile_paths)} prediction tiles for mosaicking.')

if len(pred_tile_paths) == 0:
    print('No prediction tiles found. Make sure you ran the processing loop and tiles are saved.')
else:
    src_files_to_mosaic = [rasterio.open(p) for p in pred_tile_paths]
    mosaic_array, mosaic_transform = merge(src_files_to_mosaic)  # (bands, H, W)
    # Keep profile from first tile
    mosaic_profile = src_files_to_mosaic[0].profile.copy()
    for src in src_files_to_mosaic:
        src.close()

    print("Mosaic array shape:", mosaic_array.shape)  # should be (3, H, W)

    mosaic_profile.update(
        transform=mosaic_transform,
        height=mosaic_array.shape[1],
        width=mosaic_array.shape[2],
        count=3,
        dtype='float32',
        compress='lzw',
    )

    mosaic_path = os.path.join(pred_base_dir, 'pred_windowA_mosaic_3band.tif')
    with rasterio.open(mosaic_path, 'w', **mosaic_profile) as dst:
        dst.write(mosaic_array)

    print('Mosaic saved to:', mosaic_path)
    print('Bands:')
    print('  1: dominant class (0..4)')
    print('  2: p(wetland_to_woody)')
    print('  3: uncertainty = 1 - max_prob')